# 🎯 Phase 4 — Streamlit Dashboard
**Run after:** `03_agents_pipeline_v2.ipynb`

---
## Cells
1. **Cell 1** — Install Streamlit + ngrok
2. **Cell 2** — Import & load models
3. **Cell 3** — Define pipeline function
4. **Cell 4** — Write app.py to disk
5. **Cell 5** — Launch dashboard & get public URL

## Cell 1 — Install packages

In [2]:
!pip install -q streamlit pyngrok sentence-transformers faiss-cpu xgboost shap
print('✅ Done!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 75.1 MB/s eta 0:00:00
✅ Done!


## Cell 2 — Imports & load models

In [4]:
import json, pickle, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import faiss
from sentence_transformers import SentenceTransformer
warnings.filterwarnings('ignore')

OUTPUTS_DIR = '/content'

with open(f'{OUTPUTS_DIR}/preprocessing_config.json') as f:
    config = json.load(f)
FEATURE_COLS = config['feature_cols']

with open(f'{OUTPUTS_DIR}/XGB Regressor Model.pkl', 'rb') as f:
    reg_model = pickle.load(f)
with open(f'{OUTPUTS_DIR}/XGB Classifier Model.pkl', 'rb') as f:
    cls_model = pickle.load(f)
with open(f'{OUTPUTS_DIR}/Label Encoder Model.pkl', 'rb') as f:
    le = pickle.load(f)
with open(f'{OUTPUTS_DIR}/Shap Explainer Model Training.pkl', 'rb') as f:
    shap_explainer = pickle.load(f)

print('Loading SBERT...')
sbert = SentenceTransformer('all-MiniLM-L6-v2')

faiss_index = faiss.read_index(f'{OUTPUTS_DIR}/skill_faiss.index')
with open(f'{OUTPUTS_DIR}/skill_corpus.json') as f:
    skill_corpus = json.load(f)

print('✅ All models loaded!')

Loading SBERT...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ All models loaded!


## Cell 3 — Define pipeline function

In [5]:
KNOWN_SKILLS = {
    'python','java','sql','r','scala','c++','javascript','typescript',
    'machine learning','deep learning','nlp','computer vision',
    'tensorflow','pytorch','keras','scikit-learn','xgboost','lightgbm',
    'pandas','numpy','matplotlib','seaborn','plotly',
    'spark','hadoop','kafka','airflow','dbt',
    'aws','azure','gcp','docker','kubernetes','mlflow',
    'mysql','postgresql','mongodb','redis','elasticsearch',
    'git','linux','rest api','flask','fastapi','django',
    'tableau','power bi','excel','statistics','probability',
    'data analysis','data visualization','feature engineering',
    'model deployment','ci/cd','agile','communication'
}
CRITICAL_SKILLS = {
    'python','sql','machine learning','deep learning','statistics',
    'data analysis','aws','docker','spark','tensorflow','pytorch'
}
EDU_RANK = {
    'phd':5,'doctorate':5,'m.tech':4,'msc':4,'mba':4,'master':4,
    'b.tech':3,'bsc':3,'bachelor':3,'be':3,'diploma':2
}

def extract_skills(text):
    tl = text.lower()
    found = [s for s in KNOWN_SKILLS if s in tl]
    m = re.search(r'skills?[:\s]+([^\n]+)', tl)
    if m:
        found.extend([s.strip() for s in re.split(r'[,|;]', m.group(1)) if len(s.strip()) > 2])
    return list(set(found))

def extract_exp(text):
    for p in [r'(\d+\.?\d*)\s*\+?\s*years?\s+of\s+experience',
              r'minimum\s+(\d+)\s+years?',
              r'(\d+\.?\d*)\s*\+?\s*years?\s+experience']:
        m = re.search(p, text.lower())
        if m: return float(m.group(1))
    nums = re.findall(r'(\d+\.?\d*)\s*years?', text.lower())
    return sum(float(e) for e in nums[:3]) if nums else 0.0

def extract_edu(text):
    t = text.lower()
    for deg, rank in sorted(EDU_RANK.items(), key=lambda x: -x[1]):
        if deg in t: return deg, rank
    return 'bachelor', 3

def normalize_skills(skills):
    if not skills: return []
    emb = sbert.encode([s.lower() for s in skills],
                       convert_to_numpy=True, normalize_embeddings=True).astype('float32')
    dists, idxs = faiss_index.search(emb, k=1)
    return list(set([
        skill_corpus[idxs[i][0]] if dists[i][0] > 0.75 else skills[i].lower()
        for i in range(len(skills))
    ]))

def run_pipeline(resume_text, jd_text):
    # Agent 1: Parse
    name_m  = re.search(r'^([A-Z][a-z]+ [A-Z][a-z]+)', resume_text.strip())
    title_m = re.search(r'(?:role|job title|position)[:\s]+([^\n]+)', jd_text, re.I)
    r_skills = extract_skills(resume_text)
    r_exp    = extract_exp(resume_text)
    r_deg, r_edu = extract_edu(resume_text)
    r_len    = len(resume_text.split())
    j_skills = extract_skills(jd_text)
    j_exp    = extract_exp(jd_text)
    j_deg, j_edu = extract_edu(jd_text)
    candidate = name_m.group(1) if name_m else 'Candidate'
    job_title = title_m.group(1).strip() if title_m else 'Target Role'

    # Agent 2: Normalize
    norm_r = normalize_skills(r_skills)
    norm_j = normalize_skills(j_skills)

    # Agent 3: Match
    r_set, j_set = set(norm_r), set(norm_j)
    overlap      = r_set & j_set
    skill_overlap = len(overlap) / len(j_set) if j_set else 0.0
    emb_r = sbert.encode([' '.join(norm_r)], normalize_embeddings=True)
    emb_j = sbert.encode([' '.join(norm_j)], normalize_embeddings=True)
    sbert_sim = float(np.dot(emb_r, emb_j.T)[0][0])
    exp_gap   = max(0.0, j_exp - r_exp)
    edu_match = 1 if r_edu >= j_edu else 0
    fm = {
        'feat_skill_overlap':       skill_overlap,
        'feat_resume_skill_count':  len(norm_r),
        'feat_jd_skill_count':      len(norm_j),
        'feat_missing_skill_count': len(j_set - r_set),
        'feat_edu_match':           edu_match,
        'feat_edu_level_resume':    r_edu,
        'feat_edu_level_jd':        j_edu,
        'feat_exp_years_required':  j_exp,
        'feat_resume_text_len':     r_len,
        'feat_sbert_similarity':    sbert_sim,
        'feat_resume_exp_years':    r_exp,
        'feat_exp_gap':             exp_gap
    }
    X = pd.DataFrame([[fm.get(c, 0) for c in FEATURE_COLS]], columns=FEATURE_COLS)
    match_score = float(np.clip(reg_model.predict(X)[0], 0, 1))
    label       = le.inverse_transform([cls_model.predict(X)[0]])[0]
    skill_score = round(skill_overlap * 100, 1)
    sbert_score = round(sbert_sim * 100, 1)
    exp_score   = round(max(0, 1 - exp_gap / max(j_exp, 1)) * 100, 1)

    # Agent 4: Gap
    missing_skills = [
        {'skill': s, 'importance': 'Must-have' if s in CRITICAL_SKILLS else 'Good-to-have'}
        for s in (j_set - r_set)
    ]
    missing_skills.sort(key=lambda x: 0 if x['importance'] == 'Must-have' else 1)

    # Agent 5: Explain
    shap_vals = shap_explainer.shap_values(X)
    top_pos = sorted(zip(FEATURE_COLS, shap_vals[0]), key=lambda x: x[1], reverse=True)
    top_neg = sorted(zip(FEATURE_COLS, shap_vals[0]), key=lambda x: x[1])
    pos_factors = [f.replace('feat_','').replace('_',' ') for f,v in top_pos[:3] if v > 0]
    neg_factors = [f.replace('feat_','').replace('_',' ') for f,v in top_neg[:3] if v < 0]
    missing_must = [s['skill'] for s in missing_skills if s['importance'] == 'Must-have']
    if label == 'Shortlist':   dec = f'a strong candidate for {job_title}'
    elif label == 'Maybe':     dec = f'a borderline match for {job_title}'
    else:                      dec = f'not yet ready for {job_title}'
    explanation = (
        f"{candidate} scored {round(match_score*100,1)}/100 and is {dec}. "
        f"Matching skills: {', '.join(list(overlap)[:5]) or 'none'}. "
    )
    if missing_must: explanation += f"Critical gaps: {', '.join(missing_must)}. "
    if exp_gap > 0:  explanation += f"Experience is {exp_gap:.1f} years below requirement."
    improvements = []
    if missing_must: improvements.append(f"Learn missing critical skills: {', '.join(missing_must[:3])}.")
    if exp_gap > 0:  improvements.append(f"Bridge {exp_gap:.1f}-year experience gap with projects.")
    if sbert_score < 60: improvements.append('Mirror JD keywords in your resume bullets.')
    while len(improvements) < 3:
        improvements.append('Add quantifiable achievements (e.g. improved accuracy by 15%).')
    ats_score = min(100, int(skill_score*0.5 + sbert_score*0.3 + (100 if exp_gap==0 else 50)*0.2))

    return {
        'candidate': candidate, 'job_title': job_title,
        'overall_score': round(match_score * 100, 1), 'label': label,
        'section_scores': {'Skill Match': skill_score, 'Semantic Fit': sbert_score,
                           'Experience': exp_score, 'Role Fit': round(match_score*100,1)},
        'matched_skills': list(overlap), 'missing_skills': missing_skills,
        'explanation': explanation, 'improvements': improvements[:3],
        'ats_score': ats_score, 'pos_factors': pos_factors, 'neg_factors': neg_factors,
        'shap_vals': shap_vals[0].tolist()
    }

print('✅ Pipeline function ready!')

✅ Pipeline function ready!


## Cell 4 — Write app.py to disk

In [11]:
app_code = """
import streamlit as st
import json, pickle, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import faiss
from sentence_transformers import SentenceTransformer
warnings.filterwarnings('ignore')

st.set_page_config(page_title='AI Resume Matcher', page_icon='🎯', layout='wide')

KNOWN_SKILLS = {
    'python','java','sql','r','scala','c++','javascript',
    'machine learning','deep learning','nlp','computer vision',
    'tensorflow','pytorch','keras','scikit-learn','xgboost',
    'pandas','numpy','matplotlib','seaborn','plotly',
    'spark','hadoop','kafka','airflow','aws','azure','gcp',
    'docker','kubernetes','mlflow','mysql','postgresql','mongodb',
    'git','linux','flask','fastapi','django','tableau','power bi',
    'statistics','data analysis','data visualization','feature engineering'
}
CRITICAL_SKILLS = {'python','sql','machine learning','deep learning',
                   'statistics','data analysis','aws','docker','spark',
                   'tensorflow','pytorch'}
EDU_RANK = {'phd':5,'doctorate':5,'m.tech':4,'msc':4,'mba':4,
            'master':4,'b.tech':3,'bsc':3,'bachelor':3,'be':3,'diploma':2}

@st.cache_resource
def load_models():
    with open('/content/preprocessing_config.json') as f:
        config = json.load(f)
    feature_cols = config['feature_cols']
    with open('/content/XGB Regressor Model.pkl','rb') as f: reg = pickle.load(f)
    with open('/content/XGB Classifier Model.pkl','rb') as f: cls = pickle.load(f)
    with open('/content/Label Encoder Model.pkl','rb') as f: le = pickle.load(f)
    with open('/content/Shap Explainer Model Training.pkl','rb') as f: shap_exp = pickle.load(f)
    sbert = SentenceTransformer('all-MiniLM-L6-v2')
    fi = faiss.read_index('/content/skill_faiss.index')
    with open('/content/skill_corpus.json') as f: corpus = json.load(f)
    return feature_cols, reg, cls, le, shap_exp, sbert, fi, corpus

def extract_skills(text):
    tl = text.lower()
    found = [s for s in KNOWN_SKILLS if s in tl]
    m = re.search(r'skills?[:\\s]+([^\\n]+)', tl)
    if m:
        found.extend([s.strip() for s in re.split(r'[,|;]', m.group(1)) if len(s.strip())>2])
    return list(set(found))

def extract_exp(text):
    for p in [r'(\\d+\\.?\\d*)\\s*\\+?\\s*years?\\s+of\\s+experience',
              r'minimum\\s+(\\d+)\\s+years?',
              r'(\\d+\\.?\\d*)\\s*\\+?\\s*years?\\s+experience']:
        m = re.search(p, text.lower())
        if m: return float(m.group(1))
    nums = re.findall(r'(\\d+\\.?\\d*)\\s*years?', text.lower())
    return sum(float(e) for e in nums[:3]) if nums else 0.0

def extract_edu(text):
    t = text.lower()
    for deg, rank in sorted(EDU_RANK.items(), key=lambda x: -x[1]):
        if deg in t: return deg, rank
    return 'bachelor', 3

def run_pipeline(resume_text, jd_text, feature_cols, reg, cls, le, shap_exp, sbert, fi, corpus):
    name_m  = re.search(r'^([A-Z][a-z]+ [A-Z][a-z]+)', resume_text.strip())
    title_m = re.search(r'(?:role|job title|position)[:\\s]+([^\\n]+)', jd_text, re.I)
    r_skills = extract_skills(resume_text); r_exp = extract_exp(resume_text)
    r_deg, r_edu = extract_edu(resume_text); r_len = len(resume_text.split())
    j_skills = extract_skills(jd_text); j_exp = extract_exp(jd_text)
    j_deg, j_edu = extract_edu(jd_text)
    candidate = name_m.group(1) if name_m else 'Candidate'
    job_title = title_m.group(1).strip() if title_m else 'Target Role'

    def normalize(skills):
        if not skills: return []
        emb = sbert.encode([s.lower() for s in skills], convert_to_numpy=True, normalize_embeddings=True).astype('float32')
        dists, idxs = fi.search(emb, k=1)
        return list(set([corpus[idxs[i][0]] if dists[i][0]>0.75 else skills[i].lower() for i in range(len(skills))]))

    norm_r = normalize(r_skills); norm_j = normalize(j_skills)
    r_set, j_set = set(norm_r), set(norm_j); overlap = r_set & j_set
    skill_overlap = len(overlap)/len(j_set) if j_set else 0.0
    emb_r = sbert.encode([' '.join(norm_r)], normalize_embeddings=True)
    emb_j = sbert.encode([' '.join(norm_j)], normalize_embeddings=True)
    sbert_sim = float(np.dot(emb_r, emb_j.T)[0][0])
    exp_gap = max(0.0, j_exp - r_exp); edu_match = 1 if r_edu >= j_edu else 0
    fm = {'feat_skill_overlap':skill_overlap,'feat_resume_skill_count':len(norm_r),
          'feat_jd_skill_count':len(norm_j),'feat_missing_skill_count':len(j_set-r_set),
          'feat_edu_match':edu_match,'feat_edu_level_resume':r_edu,'feat_edu_level_jd':j_edu,
          'feat_exp_years_required':j_exp,'feat_resume_text_len':r_len,
          'feat_sbert_similarity':sbert_sim,'feat_resume_exp_years':r_exp,'feat_exp_gap':exp_gap}
    X = pd.DataFrame([[fm.get(c,0) for c in feature_cols]], columns=feature_cols)
    match_score = float(np.clip(reg.predict(X)[0], 0, 1))
    label = le.inverse_transform([cls.predict(X)[0]])[0]
    skill_score = round(skill_overlap*100,1)
    sbert_score = round(sbert_sim*100,1)
    exp_score   = round(max(0,1-exp_gap/max(j_exp,1))*100,1)
    missing_skills = [{'skill':s,'importance':'Must-have' if s in CRITICAL_SKILLS else 'Good-to-have'} for s in (j_set-r_set)]
    missing_skills.sort(key=lambda x: 0 if x['importance']=='Must-have' else 1)
    shap_vals = shap_exp.shap_values(X)
    top_pos = sorted(zip(feature_cols,shap_vals[0]),key=lambda x:x[1],reverse=True)
    top_neg = sorted(zip(feature_cols,shap_vals[0]),key=lambda x:x[1])
    pos_factors = [f.replace('feat_','').replace('_',' ') for f,v in top_pos[:3] if v>0]
    neg_factors = [f.replace('feat_','').replace('_',' ') for f,v in top_neg[:3] if v<0]
    missing_must = [s['skill'] for s in missing_skills if s['importance']=='Must-have']
    if label=='Shortlist': dec=f'a strong candidate for {job_title}'
    elif label=='Maybe':   dec=f'a borderline match for {job_title}'
    else:                  dec=f'not yet ready for {job_title}'
    explanation = f"{candidate} scored {round(match_score*100,1)}/100 and is {dec}. Matching skills: {', '.join(list(overlap)[:5]) or 'none'}. "
    if missing_must: explanation += f"Critical gaps: {', '.join(missing_must)}. "
    if exp_gap > 0:  explanation += f"Experience is {exp_gap:.1f} years below requirement."
    improvements = []
    if missing_must: improvements.append(f"Learn missing critical skills: {', '.join(missing_must[:3])}.")
    if exp_gap > 0:  improvements.append(f"Bridge {exp_gap:.1f}-year experience gap with projects.")
    if sbert_score < 60: improvements.append('Mirror JD keywords in your resume bullets.')
    while len(improvements) < 3: improvements.append('Add quantifiable achievements (e.g. improved accuracy by 15%).')
    ats_score = min(100, int(skill_score*0.5 + sbert_score*0.3 + (100 if exp_gap==0 else 50)*0.2))
    return {'candidate':candidate,'job_title':job_title,'overall_score':round(match_score*100,1),
            'label':label,'section_scores':{'Skill Match':skill_score,'Semantic Fit':sbert_score,'Experience':exp_score,'Role Fit':round(match_score*100,1)},
            'matched_skills':list(overlap),'missing_skills':missing_skills,'explanation':explanation,
            'improvements':improvements[:3],'ats_score':ats_score,'pos_factors':pos_factors,
            'neg_factors':neg_factors,'shap_vals':shap_vals[0].tolist(),'feature_cols':feature_cols}

# ── UI ─────────────────────────────────────────────────────────────────────
st.title('🎯 AI Resume–Job Description Matcher')
st.markdown('*Semantic matching with explainability — SBERT + XGBoost + SHAP*')
st.divider()

with st.spinner('Loading models...'):
    feature_cols,reg,cls,le,shap_exp,sbert,fi,corpus = load_models()
st.success('✅ Models loaded!')

col1,col2 = st.columns(2)
with col1:
    st.subheader('📄 Resume')
    resume_text = st.text_area('Paste resume text here', height=300)
with col2:
    st.subheader('💼 Job Description')
    jd_text = st.text_area('Paste job description here', height=300)

st.divider()
if st.button('🚀 Analyze Match', type='primary', use_container_width=True):
    if not resume_text.strip() or not jd_text.strip():
        st.warning('Please paste both resume and job description.')
    else:
        with st.spinner('Running 5-agent pipeline...'):
            result = run_pipeline(resume_text, jd_text, feature_cols, reg, cls, le, shap_exp, sbert, fi, corpus)
        st.divider()
        score = result['overall_score']
        color = '#28a745' if score>=75 else '#ffc107' if score>=50 else '#dc3545'
        c1,c2,c3 = st.columns([1,1,2])
        with c1:
            st.markdown(f"<div style='font-size:72px;font-weight:800;text-align:center;color:{color}'>{score}</div><div style='text-align:center;color:gray'>/100</div>", unsafe_allow_html=True)
        with c2:
            badge_color = {'Shortlist':'#d4edda','Maybe':'#fff3cd','Reject':'#f8d7da'}[result['label']]
            text_color  = {'Shortlist':'#155724','Maybe':'#856404','Reject':'#721c24'}[result['label']]
            st.markdown(f"<div style='background:{badge_color};color:{text_color};padding:6px 20px;border-radius:20px;font-size:22px;font-weight:700;display:inline-block'>{result['label']}</div>", unsafe_allow_html=True)
            st.markdown(f"**ATS Score:** {result['ats_score']}/100")
        with c3:
            st.markdown(f"**Candidate:** {result['candidate']}")
            st.markdown(f"**Role:** {result['job_title']}")
            st.markdown('**Strengths:** ' + (', '.join(result['pos_factors']) or '—'))
            st.markdown('**Weaknesses:** ' + (', '.join(result['neg_factors']) or '—'))
        st.divider()
        st.subheader('📊 Section-wise Scores')
        scores = result['section_scores']
        fig,ax = plt.subplots(figsize=(10,3))
        colors2 = ['#28a745' if v>=75 else '#ffc107' if v>=50 else '#dc3545' for v in scores.values()]
        bars = ax.barh(list(scores.keys()), list(scores.values()), color=colors2, edgecolor='none', height=0.5)
        ax.set_xlim(0,100)
        ax.axvline(75,color='green',linestyle='--',linewidth=1,alpha=0.5)
        ax.axvline(50,color='orange',linestyle='--',linewidth=1,alpha=0.5)
        for bar,val in zip(bars,scores.values()): ax.text(bar.get_width()+1,bar.get_y()+bar.get_height()/2,f'{val}/100',va='center',fontsize=11)
        ax.spines[['top','right']].set_visible(False)
        plt.tight_layout(); st.pyplot(fig); plt.close()
        st.divider()
        c1,c2 = st.columns(2)
        with c1:
            st.subheader('✅ Matched Skills')
            if result['matched_skills']:
                st.markdown(''.join([f"<span style='background:#d4edda;color:#155724;padding:4px 10px;border-radius:12px;font-size:13px;margin:3px;display:inline-block'>{s}</span>" for s in result['matched_skills']]), unsafe_allow_html=True)
            else: st.info('No exact matches found.')
        with c2:
            st.subheader('❌ Missing Skills')
            if result['missing_skills']:
                tags = ''.join([f"<span style='background:{'#f8d7da' if s['importance']=='Must-have' else '#fff3cd'};color:{'#721c24' if s['importance']=='Must-have' else '#856404'};padding:4px 10px;border-radius:12px;font-size:13px;margin:3px;display:inline-block'>{s['skill']} ({s['importance']})</span>" for s in result['missing_skills']])
                st.markdown(tags, unsafe_allow_html=True)
            else: st.success('No skill gaps!')
        st.divider()
        st.subheader('🔍 SHAP Explainability — Why This Score?')
        sv = result['shap_vals']; fn = [f.replace('feat_','').replace('_',' ') for f in result['feature_cols']]
        sp = sorted(zip(fn,sv), key=lambda x: abs(x[1]), reverse=True)[:8]
        ns,vs = zip(*sp)
        fig2,ax2 = plt.subplots(figsize=(10,4))
        ax2.barh(ns,vs,color=['#28a745' if v>0 else '#dc3545' for v in vs],edgecolor='none',height=0.5)
        ax2.axvline(0,color='black',linewidth=0.8); ax2.set_xlabel('SHAP value')
        ax2.set_title('Green = helped score | Red = hurt score')
        ax2.spines[['top','right']].set_visible(False)
        plt.tight_layout(); st.pyplot(fig2); plt.close()
        st.divider()
        c1,c2 = st.columns(2)
        with c1:
            st.subheader('📝 Explanation'); st.info(result['explanation'])
        with c2:
            st.subheader('💡 Improvement Tips')
            for i,tip in enumerate(result['improvements'],1): st.markdown(f'**{i}.** {tip}')
        st.divider()
        save = {k:v for k,v in result.items() if k not in ['shap_vals','feature_cols']}
        st.download_button('⬇️ Download Report (JSON)', data=json.dumps(save,indent=2,default=str),
            file_name=f"report_{result['candidate'].replace(' ','_')}.json",
            mime='application/json', use_container_width=True)
"""

with open('/content/app.py', 'w') as f:
    f.write(app_code)
print('✅ app.py written!')



✅ app.py written!


## Cell 5 — Launch dashboard & get public URL

In [12]:
import subprocess, threading, time
from pyngrok import ngrok

subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
time.sleep(2)

def run_streamlit():
    subprocess.run([
        'streamlit', 'run', '/content/app.py',
        '--server.port=8501',
        '--server.headless=true',
        '--server.enableCORS=false',
        '--server.enableXsrfProtection=false'
    ])

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()
time.sleep(6)

ngrok.set_auth_token("3BGIOKA56HwNMYOsZRd8icjNmPO_5TwMTYvKfofeydnSnqBEA")  # ← add this line
public_url = ngrok.connect(8501)
print('=' * 55)
print('🎯 DASHBOARD IS LIVE!')
print('=' * 55)
print(f'👉 Click this URL: {public_url}')

🎯 DASHBOARD IS LIVE!
👉 Click this URL: NgrokTunnel: "https://undolorously-glyptic-alfredia.ngrok-free.dev" -> "http://localhost:8501"
